In [ ]:
import boto3

session = boto3.Session(profile_name="mehrnaz-dev", region_name="ap-southeast-2")

In [ ]:
client = session.client("prompt_eval")

model_id="au.anthropic.claude-haiku-4-5-20251001-v1:0"

def add_message(messages , text , role):
    new_message = {
        'role' : role ,
         'content' : [
             {
                 'text' : text
             }
         ]
    }
    messages.append(new_message)
    
def chat(messages , sys_prompt = "" , temp = 0.1 , stop = []):
    
    params = {
        "modelId": model_id,
        "messages" : messages,
        "inferenceConfig" : {
            "temperature" : temp,
            "stopSequences": stop 
         }
        
    }
     
    if sys_prompt :
        params["system"] = [{"text" : sys_prompt}]
        
    response = client.converse( **params )
          
    
    return response['output']['message']['content'][0]['text']

def chat_stream(messages , sys_prompt = "" , temp = 0.1):
    
    params = {
        "modelId": model_id,
        "messages" : messages,
        "inferenceConfig" : {
            "temperature" : temp
         }
        
    }
     
    if sys_prompt :
        params["system"] = [{"text" : sys_prompt}]
        
    response = client.converse_stream( **params )
    
    
    answer = []
    
    for event in response["stream"] : 
                if "contentBlockDelta" in event :
                    chunk = event["contentBlockDelta"]["delta"]["text"]
                    print(chunk , end="")
                    answer.append(chunk)
                    
    full_chunk = "".join(answer)
   
    
    return full_chunk



def conversation(messages , sys_prompt = "" , temp = 0.1):
    print("Assistant >> Welcome to AWS help Desk! ")
    while True:
        user_input = input("You >>").strip()
        
        print("You >> " , user_input , "\n" )
        
        if user_input.lower() in ['exit' , 'quit' , 'q' , 'e' ] :
            print("Assistant >> Bye!")
            break
        
            
        add_message( messages , user_input , 'user')
        
        
        assistant_reponse= chat( messages , sys_prompt , temp)
        
        add_message(messages , assistant_reponse , 'assistant')
        
        print(f'Assistant >> {assistant_reponse}\n')


In [ ]:

import json 

def generate_dataset():
    prompt = """
    Generate 3 AWS-related tasks that require Python, JSON, or Regex solutions.
    
    Focus on tasks that can be solved by writing a single Python function, 
    a single JSON object, or tasks that do not require writing much code.
    
    Example output:
    [
        {
            "task": "Description of task"
            "format" : JSON or Python or Regex 
        },
        
    ]
    
    Please generate 3 objects.
    """
    messages = []
    add_message(messages , prompt , 'user')
    add_message(messages , "```json" , 'assistant')
    output = chat(messages , stop=["```"])
    return json.loads(output)
    




# prompt eval workflow 

# prompt draft

# eval set ( possible inputs that we might want to put into our prompt)

# feed through claude to get actual answer

# grade them ( using grader ) -- gives number base on quality of answer

# make average of the scores (to say how well)

# iterate for number to increase


# eval workflow for use cases in aws --> output should be : Python , JSON or a regular expression with no explanation

# now we have to assemble a dataset 




In [ ]:
dataset = generate_dataset()

with open("dataset.json" , "w") as f:
    json.dump(dataset,f , indent=2)

In [ ]:
def run_prompt(test_case):
    prompt = f"""
    Provide a solution to the following task:
      {test_case["task"]}
      
    * only provide answer in python or json or regex
    * do not add additional comments or commentry 
    """
    messages = []
    add_message(messages , prompt , "user")
    add_message(messages, "```code" , "assistant")
    response =  chat(messages , stop="```")
    return response
    

In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f""" 
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    messages = []
    add_message(messages , eval_prompt , "user")
    add_message(messages ,"```json" , "assistant" )
    
    eval_res = chat(messages , stop=["```"])
    return json.loads(eval_res)
    

: 

In [ ]:
import ast
import re


def validate_json(text) : 
    try:
        json.load(text)
        return True
    except (ValueError , TypeError):
        return False
    
def validate_python(text):
    try:
        ast.parse(text)
        return True
    except SyntaxError:
        return False

def validate_regext(text):
    try:
        re.compile(text)
        return True
    except re.error:
        return False
    

def grade_code(test_case):
    
    
    



In [ ]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case , output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    
    return {
        "output": output,
        "test_case" : test_case,
        "score" : score,
        "reasoning" : reasoning
    }
    
    

In [ ]:

from statistics import mean

def run_eval(dataset):
    
    eval_results = []
    
    for value in dataset:
        res = run_test_case(value)
        eval_results.append(res)
    
    avg_score = mean([result["score"] for result in eval_results])
    
    print(f"Average Score : {avg_score}")
    
    return eval_results
        
        
        

In [ ]:
with open("dataeset.json" , "r") as file :
        dataset = json.load(file)
        
results = run_eval(dataset)

In [ ]:
print(json.dumps(results , indent=2))